# ALM frictionless mortar contact condition (scalar multiplier)

This notebook generates `custom_conditions/ALM_frictionless_mortar_contact_condition.cpp`, the local left-
and right-hand sides of `AugmentedLagrangianMethodFrictionlessMortarContactCondition<TDim, TNumNodes, TNormalVariation, TNumNodesMaster>`,
the **augmented Lagrangian frictionless** mortar contact condition with a **scalar** Lagrange multiplier
(`LAGRANGE_MULTIPLIER_CONTACT_PRESSURE`, one DoF per slave node). Theory: thesis §4.3.3 and the
[Frictionless contact](https://kratosmultiphysics.github.io/Kratos/pages/Applications/Contact_Structural_Mechanics_Application/Theory/Frictionless_Contact.html)
page of the documentation.

## Formulation

### Augmented Lagrangian (thesis eqs. 4.9-4.13)

The augmented Lagrangian of Alart and Curnier reformulates the contact law as equations without inequalities:

$$\mathcal{L}_{co}(\mathbf{u}, \lambda_n) = \int_{\Gamma_c^{(1)}} \begin{cases} k \lambda_n\, g_n + \dfrac{\varepsilon}{2} g_n^2 & \bar{\lambda}_n \le 0 \quad \text{(contact zone)} \\[4pt] -\dfrac{k}{2\varepsilon} \lambda_n^2 & \bar{\lambda}_n > 0 \quad \text{(gap zone)} \end{cases} \mathrm{d}\Gamma, \qquad \bar{\lambda}_n = k\,\lambda_n + \varepsilon\, g_n$$

with $\varepsilon$ the penalty parameter (`PenaltyParameter`, nodal `INITIAL_PENALTY`) and $k$ the scale factor
(`ScaleFactor`, `SCALE_FACTOR`); $\bar{\lambda}_n$ is the **augmented normal pressure**
(`AUGMENTED_NORMAL_CONTACT_PRESSURE`). Its variation is the weak form (thesis eq. 4.13)

$$\delta\mathcal{L}_{co} = \int_{\Gamma_c^{(1)}} \begin{cases} \bar{\lambda}_n\, \delta g_n + k\, g_n\, \delta\lambda_n & \bar{\lambda}_n \le 0 \\[4pt] -\dfrac{k^2}{\varepsilon} \lambda_n\, \delta\lambda_n & \bar{\lambda}_n > 0 \end{cases} \mathrm{d}\Gamma$$

In the contact zone the multiplier equation enforces the gap weakly and the displacement equation carries the
augmented pressure; in the gap zone the multiplier is simply driven to zero, which keeps the system square
when the active set changes. The switch depends on the sign of $\bar{\lambda}_n$ only (the active-set test of
`ActiveSetUtilities`).

### Discrete form (thesis eqs. 4.29-4.31, 4.35)

With the dual mortar operators $\mathbf{D}$ (diagonal) and $\mathbf{M}$ and the current coordinates
$\mathbf{x}^{(i)} = \mathbf{X}^{(i)} + \mathbf{u}^{(i)}$, the nodal weighted gap and augmented pressure are

$$\tilde{g}_{n,j} = -\,\mathbf{n}_j \cdot \left( \mathbf{D}\,\mathbf{x}^{(1)} - \mathbf{M}\,\mathbf{x}^{(2)} \right)_j, \qquad \bar{\lambda}_{n,j} = k\,\lambda_{n,j} + \varepsilon_j\, \tilde{g}_{n,j}$$

and the multiplier residuals of thesis eq. 4.35 are $\mathbf{r}_{\lambda_{\mathcal{A}}} = -k\,\mathbf{n} \cdot (\mathbf{D}\mathbf{x}_1 - \mathbf{M}\mathbf{x}_2)$
for active nodes and $\mathbf{r}_{\lambda_\mathcal{I}} = \frac{k^2}{\varepsilon}\lambda_n$ for inactive ones, the multiplier
columns of the displacement rows being $k(\mathbf{n}\cdot\mathbf{D})^T$ and $-k(\mathbf{n}\cdot\mathbf{M})^T$.

### The two generated branches

Per slave node $j$, with $\mathcal{D}_j$ the `DynamicFactor`:

| branch | $\mathcal{R}_j$ |
|---|---|
| active | $\mathcal{D}_j\, \bar{\lambda}_{n,j}\, \mathbf{n}_j \cdot \left( \mathbf{D}\,\mathbf{w}^{(1)} - \mathbf{M}\,\mathbf{w}^{(2)} \right)_j + k\, \tilde{g}_{n,j}\, \delta\lambda_{n,j}$ |
| inactive | $-\dfrac{k^2}{\varepsilon_j} \lambda_{n,j}\, \delta\lambda_{n,j}$ |

The first term is $\bar{\lambda}_n \delta g_n$ in discrete form (with the sign convention of the Kratos residual),
the second is $k\, g_n \delta\lambda_n$ and the inactive branch is the gap-zone term. The run-time dispatch
is, per node, `if (r_geometry[i].IsNot(ACTIVE)) {...} else {...}`.

## From the functional to the generated C++

All the mechanics of the generation live in `../mortar_condition_generator.py` (see also the
[Automatic differentiation](https://kratosmultiphysics.github.io/Kratos/pages/Applications/Contact_Structural_Mechanics_Application/Theory/Automatic_Differentiation.html)
page, thesis Appendix C):

1. **Symbols** (`SymbolSet`): the nodal unknowns `u1`, `u2` (displacements of the slave and master nodes),
   the multipliers, the test functions `w1`, `w2`, `wLM`, the reference coordinates `X1`, `X2`, the nodal
   normals `NormalSlave`, the mortar operators `DOperator`, `MOperator` and the parameters. The current
   coordinates are $\mathbf{x}^{(i)} = \mathbf{X}^{(i)} + \mathbf{u}^{(i)}$ and the nodal **weighted gap**
   (thesis eq. 4.31) is
   $$\tilde{g}_{n,j} = -\,\mathbf{n}_j \cdot \left( \mathbf{D}\, \mathbf{x}^{(1)} - \mathbf{M}\, \mathbf{x}^{(2)} \right)_j$$
   which is **positive for an open gap** and negative for penetration (`NormalGap` / `WEIGHTED_GAP`).
2. **AD exceptions** (thesis §C.3.1): $\mathbf{D}$, $\mathbf{M}$ and, when `TNormalVariation` is `true`, $\mathbf{n}$
   are not expressed in terms of the displacements. They are declared *undefined functions of the DoFs*
   (`DefineDofDependencyMatrix`), so that the chain rule produces unevaluated derivatives that are mapped to
   the arrays computed at run time by `DerivativesUtilities`:

   | symbolic node | C++ |
   |---|---|
   | `DOperator_i_j(u...)` | `DOperator(i,j)` |
   | `Derivative(DOperator_i_j(u...), u_k)` | `DeltaDOperator[k](i,j)` |
   | `Derivative(MOperator_i_j(u...), u_k)` | `DeltaMOperator[k](i,j)` |
   | `Derivative(NormalSlave_i_j(u...), u_k)` | `DeltaNormalSlave[k](i,j)` (normal variation only) |

   The index `k` runs over the slave displacement DoFs first and then the master ones, the ordering used by
   `MortarOperatorWithDerivatives`.
3. **Differentiation**: for every slave node $j$ and every active-set branch the functional $\mathcal{R}_j$
   returned by the function below is differentiated: $\mathbf{r} = \partial \mathcal{R} / \partial \mathbf{w}$
   (local RHS) and $\mathbf{K} = -\partial \mathbf{r} / \partial \mathbf{d}$ (local LHS), with the DoF vector
   $\mathbf{d} = [\mathbf{u}^{(2)}, \mathbf{u}^{(1)}, \boldsymbol{\lambda}]$ ordered *master, slave, multiplier* exactly as
   `GetDofList`. This is the Kratos convention $\mathbf{K}\,\Delta\mathbf{d} = \mathbf{r}$.
4. **Printing**: the derivative nodes are replaced by plain symbols, `sympy.cse` collects the common factors
   (`clhs*`, `crhs*`) and `sympy.ccode` prints C++; only the non-zero entries are emitted, accumulated with `+=`.
5. **Assembly of the file**: one `CalculateLocalLHS` specialisation per geometry pair (`2D2N`, `3D3N`, `3D4N`,
   `3D3N4N`, `3D4N3N`) and per `TNormalVariation` value; the RHS does **not** depend on the derivatives of the
   normal, so `StaticCalculateLocalRHS` is generated only for `TNormalVariation = false` and the `true`
   specialisation forwards to it. The bodies are substituted into the `*_template.cpp` of this folder at the
   `// replace_lhs` / `// replace_rhs` markers and the result is written once.

**Sign convention of the test-function quantities.** Every `<quantity>w` symbol (`NormalwGap`, `TangentwSlip*`)
is *minus* the variation of the quantity in the direction of the test functions, $X_w = -\delta X$: with the
gap defined as above, `NormalwGap = +n.(D w1 - M w2)`, so that the virtual work of a traction $\mathbf{t}$
is written $\mathbf{t} \cdot X_w$ and the residual is $-\delta\Pi$ (the force acting on the bodies).

## How to run this notebook

* **Requirements**: Python 3 and `sympy` (any modern version, tested with 1.14). A compiled Kratos is *not*
  needed: the shared module `../mortar_condition_generator.py` imports `custom_sympy_fe_utilities.py` and the
  core `sympy_fe_utilities.py` directly from the source tree.
* **Interactively**: open it with Jupyter from this folder and run all cells.
* **Headless** (no Jupyter installed): `python3 ../run_notebook.py <this notebook>` executes the code cells
  with the standard library only.
* The equivalent command-line script `generate_*.py` in this folder contains the *same* functional and
  generation call; keep both in sync when the formulation changes.

The output is written directly into `custom_conditions/` (overwriting the committed file). The generation of
the five geometries and the two normal-variation flags takes from a few minutes (frictionless) to about an
hour (ALM frictional). Restrict `COMBINATIONS` / `NORMAL_VARIATIONS` in the configuration cell for a quick
test, or set `OUTPUT_DIR` to a scratch folder.

In [ ]:
import os
import sys

# The shared generator module lives one folder up (automatic_differentiation/)
sys.path.insert(0, os.path.abspath(".."))
import mortar_condition_generator as generator

print("sympy", generator.sympy.__version__)

## Symbols of `SymbolSet` used by the functional

| symbol | meaning | C++ counterpart |
|---|---|---|
| `s.LMNormal[j]`, `s.wLMNormal[j]` | $\lambda_{n,j}$ and its test function | `LAGRANGE_MULTIPLIER_CONTACT_PRESSURE` |
| `s.NormalGap[j]` | $\tilde{g}_{n,j}$ | `WEIGHTED_GAP` |
| `s.NormalSlave.row(j)` | $\mathbf{n}_j$ | `NORMAL` |
| `s.Dw1Mw2.row(j)` | $(\mathbf{D}\,\mathbf{w}^{(1)} - \mathbf{M}\,\mathbf{w}^{(2)})_j$ | — |
| `s.ScaleFactor`, `s.PenaltyParameter[j]`, `s.DynamicFactor[j]` | $k$, $\varepsilon_j$, $\mathcal{D}_j$ | `SCALE_FACTOR`, `INITIAL_PENALTY`, `DYNAMIC_FACTOR` |

The branch identifiers passed to the functional are `inactive` and `active`.

## The functional

This is the physics of the condition; it is the only family-specific input of the generator.

In [ ]:
def frictionless_functional(s, node, branch):
    """Galerkin functional of one slave node (thesis eqs. 4.36-4.44). ``branch`` is ``inactive`` or ``active``."""
    rv_galerkin = 0
    if branch == "active":
        augmented_contact_pressure = (s.ScaleFactor * s.LMNormal[node] + s.PenaltyParameter[node] * s.NormalGap[node])
        rv_galerkin += s.DynamicFactor[node] * (augmented_contact_pressure * s.NormalSlave.row(node)).dot(s.Dw1Mw2.row(node))
        rv_galerkin += s.ScaleFactor * s.NormalGap[node] * s.wLMNormal[node]
    else:
        rv_galerkin -= s.ScaleFactor**2 / s.PenaltyParameter[node] * s.LMNormal[node] * s.wLMNormal[node]
    return rv_galerkin

## Configuration

`COMBINATIONS` holds the `(dimension, slave nodes, master nodes)` triplets to generate and
`NORMAL_VARIATIONS` the values of `TNormalVariation`. The output goes to `custom_conditions/` by default.

In [ ]:
COMBINATIONS = generator.DEFAULT_COMBINATIONS   # ((2, 2, 2), (3, 3, 3), (3, 4, 4), (3, 3, 4), (3, 4, 3))
NORMAL_VARIATIONS = (False, True)
TEMPLATE_DIR, OUTPUT_DIR = generator.DefaultDirectories(os.getcwd())
print("template folder:", TEMPLATE_DIR)
print("output folder  :", OUTPUT_DIR)

## Generation

Every branch reports the size of the local system and the number of non-zero entries. The generated
code is checked for symbolic leftovers before the file is written.

In [ ]:
output_path = generator.Generate(generator.ALM_FRICTIONLESS, frictionless_functional, TEMPLATE_DIR, OUTPUT_DIR, COMBINATIONS, NORMAL_VARIATIONS)

## Check of the generated file

The file must contain, for each of the geometries and normal-variation flags requested, one
`CalculateLocalLHS` specialisation and one `StaticCalculateLocalRHS` specialisation (a full body for
`false`, a forwarder for `true`).

In [ ]:
import re

with open(output_path) as generated_file:
    generated = generated_file.read()

lhs_specialisations = re.findall(r"^void AugmentedLagrangianMethodFrictionlessMortarContactCondition<(\d+),\s*(\d+), (true|false), (\d+)>::CalculateLocalLHS\(", generated, re.MULTILINE)
rhs_specialisations = re.findall(r"^void AugmentedLagrangianMethodFrictionlessMortarContactCondition<(\d+),\s*(\d+), (true|false), (\d+)>::StaticCalculateLocalRHS\(", generated, re.MULTILINE)
expected = len(COMBINATIONS) * len(NORMAL_VARIATIONS)
print("{} lines, {} LHS and {} RHS specialisations (expected {} each)".format(generated.count("\n"), len(lhs_specialisations), len(rhs_specialisations), expected))
assert len(lhs_specialisations) == expected and len(rhs_specialisations) == expected
assert "Derivative(" not in generated and "//subsvar_" not in generated